# 02 - Preprocessing

## 1. Load & build datetime

In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "Bangladesh_Multi_Site_Air_Quality.csv"

df = pd.read_csv("../Bangladesh_Multi_Site_Air_Quality.csv")
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)
df.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime
0,173641,2022,8,4,0,7.4,11.6,0.4,2.0,110.0,28.0,26.3,1003.8,25.0,0.0,SE,3.68,Barisal,2022-08-04 00:00:00
1,173642,2022,8,4,1,5.0,8.1,0.4,2.0,108.0,28.0,26.8,1004.8,24.8,0.1,SE,3.70,Barisal,2022-08-04 01:00:00
2,173643,2022,8,4,2,4.7,7.5,0.4,2.1,105.0,28.0,28.2,1004.5,25.0,0.0,SE,4.56,Barisal,2022-08-04 02:00:00
3,173644,2022,8,4,3,4.9,7.7,0.4,2.1,101.0,29.0,28.9,1004.8,25.1,0.0,SE,4.20,Barisal,2022-08-04 03:00:00
4,173645,2022,8,4,4,5.2,8.1,0.4,1.7,99.0,32.0,30.1,1004.6,25.4,0.2,SE,4.18,Barisal,2022-08-04 04:00:00


## 2. Check for missing hourly timestamps per station\nEssential before building lag/rolling features - gaps would silently corrupt them.

In [3]:
print("Checking for timestamp gaps per station...")
for station, g in df.groupby("station"):
    full_range = pd.date_range(g["datetime"].min(), g["datetime"].max(), freq="h")
    missing = full_range.difference(g["datetime"])
    print(f"  {station}: {len(missing)} missing hourly timestamps out of {len(full_range)}")

# NOTE: if gaps exist, reindex per station and interpolate, e.g.:
# g = g.set_index("datetime").reindex(full_range).interpolate(limit=3)
# Not needed here since this dataset has none in practice.

Checking for timestamp gaps per station...
  Barisal: 0 missing hourly timestamps out of 34728
  Chittagong: 0 missing hourly timestamps out of 34728
  Dhaka: 0 missing hourly timestamps out of 34728
  Khulna: 0 missing hourly timestamps out of 34728
  Rajshahi: 0 missing hourly timestamps out of 34728
  Sylhet: 0 missing hourly timestamps out of 34728


## 3. Duplicate check

In [4]:
dupes = df.duplicated(subset=["station", "datetime"]).sum()
print(f"Duplicate (station, datetime) rows: {dupes}")
df = df.drop_duplicates(subset=["station", "datetime"])

Duplicate (station, datetime) rows: 0


## 4. Outlier capping (winsorize)
Cap pollutant/meteorological columns at the 1st/99th percentile **per station**.
We deliberately do NOT touch PM2.5's upper tail - that's our real spike signal,
capping it would destroy the thing we're trying to predict.

In [5]:
cap_cols = ["PM10", "SO2", "NO2", "CO", "O3", "WSPM"]

def winsorize_group(g):
    for col in cap_cols:
        lo, hi = g[col].quantile([0.01, 0.99])
        g[col] = g[col].clip(lo, hi)
    return g

# NOTE: must pass df.columns explicitly - newer pandas groupby().apply()
# silently drops the grouping column ("station") otherwise.
df = df.groupby("station", group_keys=False)[df.columns].apply(winsorize_group)

non_negative_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "WSPM", "RAIN"]
for col in non_negative_cols:
    n_neg = (df[col] < 0).sum()
    if n_neg > 0:
        print(f"WARNING: {n_neg} negative values in {col}, clipping to 0")
        df[col] = df[col].clip(lower=0)
print("Winsorizing done.")

Winsorizing done.


## 5. Time-based train/test split
**Important:** never split randomly for time series data - it leaks future
information into training. Train = before 2026-01-01, Test = 2026 onward.

In [6]:
SPLIT_DATE = "2026-01-01"
train = df[df["datetime"] < SPLIT_DATE].copy()
test = df[df["datetime"] >= SPLIT_DATE].copy()

print(f"Train: {train.shape[0]} rows ({train['datetime'].min()} -> {train['datetime'].max()})")
print(f"Test:  {test.shape[0]} rows ({test['datetime'].min()} -> {test['datetime'].max()})")

train.to_csv("train_clean.csv", index=False)
test.to_csv("test_clean.csv", index=False)
print("Saved train_clean.csv and test_clean.csv")

Train: 179424 rows (2022-08-04 00:00:00 -> 2025-12-31 23:00:00)
Test:  28944 rows (2026-01-01 00:00:00 -> 2026-07-20 23:00:00)
Saved train_clean.csv and test_clean.csv
